# 🎯 SAM3 + Gemma 4 — Auto-Labeling Pipeline

> **Facebook SAM3** tüm nesne adaylarını segmentler →  
> **Google Gemma 4** hangi adayların hedef sınıf olduğunu JSON ile seçer →  
> YOLO formatında `.txt` etiketler üretilir

## Kaggle Secrets (Add-ons → Secrets)
| Key | Value |
|-----|-------|
| `HF_TOKEN` | HuggingFace Read token |

## HuggingFace Lisans Onayları
Çalıştırmadan önce her ikisine de gir ve **Access repository** butonuna bas:
- https://huggingface.co/facebook/sam3
- https://huggingface.co/google/gemma-4-E2B-it

## Kaggle Dataset Input
Etiketlenecek görüntüleri **Add Input → Datasets** ile ekle

In [ ]:
# ── 1. Kurulum ──────────────────────────────────────────────────────────────
# transformers 5.5.0 — Sam3Model ve Gemma4ForConditionalGeneration burada
!pip install -q "transformers==5.5.0" accelerate pillow

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'Device : {DEVICE}')
print(f'Dtype  : {DTYPE}')
if DEVICE == 'cuda':
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── 2. HuggingFace Token ─────────────────────────────────────────────────────
import os
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    print('✅ HF_TOKEN Kaggle Secrets\'ten alındı')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    print('⚠️  Kaggle Secret bulunamadı, env var deneniyor')

if not HF_TOKEN:
    raise ValueError('HF_TOKEN boş! Add-ons → Secrets → HF_TOKEN ekle')

# huggingface_hub login
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('HuggingFace login OK ✅')

In [ ]:
# ── 3. Konfigürasyon ─────────────────────────────────────────────────────────
from pathlib import Path

# ---------------------------------------------------------------------------
# HEDEF SINIFLAR — Gemma 4'ün arayacağı nesneler
# ---------------------------------------------------------------------------
TARGET_CLASSES = [
    'tank',
    'armored_vehicle',
    'military_truck',
    'artillery',
    'helicopter',
    'soldier',
]
CLASS_MAP  = {i: name for i, name in enumerate(TARGET_CLASSES)}
NAME_TO_ID = {name: i for i, name in CLASS_MAP.items()}

# ---------------------------------------------------------------------------
# MODEL ID'LERİ
# ---------------------------------------------------------------------------
SAM3_MODEL_ID  = 'facebook/sam3'
GEMMA_MODEL_ID = 'google/gemma-4-E2B-it'

# ---------------------------------------------------------------------------
# SAM3 PARAMETRELERİ
# ---------------------------------------------------------------------------
SAM_CONF_THRESHOLD = 0.5
SAM_MASK_THRESHOLD = 0.5
MIN_AREA_RATIO     = 0.001
MAX_AREA_RATIO     = 0.85
MAX_CANDIDATES     = 30

# ---------------------------------------------------------------------------
# GİRDİ / ÇIKTI
# ---------------------------------------------------------------------------
IMAGE_INPUT_DIR  = ''   # boş = otomatik bul
OUTPUT_DIR       = Path('/kaggle/working/sam3_labeled')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

print(f'Sınıflar: {TARGET_CLASSES}')
print('Config OK ✅')

In [ ]:
# ── 4. Modelleri Yükle ───────────────────────────────────────────────────────
from transformers import (
    Sam3Model, Sam3Processor,
    Gemma4ForConditionalGeneration, AutoProcessor
)

# ── SAM3 ────────────────────────────────────────────────────────────────────
print(f'SAM3 yükleniyor: {SAM3_MODEL_ID}')
sam_processor = Sam3Processor.from_pretrained(SAM3_MODEL_ID, token=HF_TOKEN)
sam_model     = Sam3Model.from_pretrained(SAM3_MODEL_ID, token=HF_TOKEN).to(DEVICE).eval()
print(f'  SAM3 hazır — {sum(p.numel() for p in sam_model.parameters())/1e6:.0f}M parametre ✅')

# ── Gemma 4 ─────────────────────────────────────────────────────────────────
print(f'Gemma 4 yükleniyor: {GEMMA_MODEL_ID}')
gemma_processor = AutoProcessor.from_pretrained(GEMMA_MODEL_ID, token=HF_TOKEN)
gemma_model     = Gemma4ForConditionalGeneration.from_pretrained(
    GEMMA_MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=DTYPE,
    device_map='auto'
).eval()
print('  Gemma 4 hazır ✅')

if DEVICE == 'cuda':
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'\nGPU bellek: {used:.1f} / {total:.1f} GB')

In [ ]:
# ── 5. Görüntü Dizinini Bul ──────────────────────────────────────────────────
import os

def find_image_dir(base='/kaggle/input'):
    best_dir, best_count = Path(base), 0
    for dirpath, _, filenames in os.walk(base):
        imgs = [f for f in filenames if Path(f).suffix.lower() in IMAGE_EXTENSIONS]
        if len(imgs) > best_count:
            best_count, best_dir = len(imgs), Path(dirpath)
    return best_dir, best_count

img_dir = Path(IMAGE_INPUT_DIR) if IMAGE_INPUT_DIR else find_image_dir()[0]
all_images = sorted(p for p in img_dir.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS)
print(f'📂 Dizin   : {img_dir}')
print(f'🖼️  Görüntü : {len(all_images):,}')

In [ ]:
# ── 6. Çıktı Klasörleri ──────────────────────────────────────────────────────
img_out   = OUTPUT_DIR / 'images'
label_out = OUTPUT_DIR / 'labels'
img_out.mkdir(parents=True, exist_ok=True)
label_out.mkdir(parents=True, exist_ok=True)
print(f'Çıktı: {OUTPUT_DIR}')

In [ ]:
# ── 7. SAM3 — Aday Maske Üretimi ─────────────────────────────────────────────
import json
import numpy as np
from PIL import Image

def run_sam3(image: Image.Image) -> list:
    W, H = image.size

    inputs = sam_processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = sam_model(**inputs)

    processed = sam_processor.post_process_instance_segmentation(
        outputs,
        threshold=SAM_CONF_THRESHOLD,
        mask_threshold=SAM_MASK_THRESHOLD,
        target_sizes=[(H, W)]
    )[0]

    candidates = []
    segments = processed.get('segments_info', [])
    masks    = processed.get('segmentation', None)

    for seg in segments:
        sid  = seg['id']
        if masks is None: continue
        mask = (masks == sid).cpu().numpy()

        rows = mask.any(axis=1); cols = mask.any(axis=0)
        if not rows.any(): continue
        y1 = int(rows.argmax())
        y2 = int(H - rows[::-1].argmax() - 1)
        x1 = int(cols.argmax())
        x2 = int(W - cols[::-1].argmax() - 1)

        area_ratio = ((x2-x1) * (y2-y1)) / (W*H)
        if area_ratio < MIN_AREA_RATIO or area_ratio > MAX_AREA_RATIO: continue

        candidates.append({
            'bbox' : [x1, y1, x2, y2],
            'score': float(seg.get('score', 1.0)),
        })

    candidates.sort(key=lambda x: -x['score'])
    return candidates[:MAX_CANDIDATES]

print('SAM3 fonksiyonu hazır ✅')

In [ ]:
# ── 8. Gemma 4 — Sınıflandırma ───────────────────────────────────────────────
import ast, re

SYSTEM_PROMPT = """You are a precise military object detection assistant.
Identify which candidate regions contain military objects.
Always respond with valid JSON only — no markdown, no explanation."""

def run_gemma4(image: Image.Image, candidates: list) -> list:
    if not candidates:
        return []

    W, H = image.size
    bbox_list = [c['bbox'] for c in candidates]

    user_text = f"""Aerial image ({W}x{H}px). SAM3 found {len(candidates)} regions:
{json.dumps(bbox_list)}

Target classes: {', '.join(TARGET_CLASSES)}

Return ONLY JSON:
{{"detections": [{{"index": 0, "class": "tank"}}, {{"index": 2, "class": "soldier"}}]}}
If nothing matches: {{"detections": []}}"""

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",   "content": [{"type": "image", "image": image},
                                        {"type": "text",  "text": user_text}]},
    ]

    inputs = gemma_processor.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors='pt'
    ).to(gemma_model.device)

    with torch.no_grad():
        out_ids = gemma_model.generate(**inputs, max_new_tokens=256,
                                       temperature=0.1, do_sample=False)

    raw = gemma_processor.decode(
        out_ids[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True
    ).strip()
    raw = re.sub(r'^```[\w]*\n?|```$', '', raw, flags=re.MULTILINE).strip()

    try:    parsed = json.loads(raw)
    except: 
        try:    parsed = ast.literal_eval(raw)
        except: print(f'  ⚠️ JSON hatası: {raw[:60]}'); return []

    results = []
    for det in parsed.get('detections', []):
        idx      = det.get('index')
        cls_name = det.get('class', '').lower().replace(' ', '_')
        if idx is None or idx >= len(candidates): continue
        if cls_name not in NAME_TO_ID:            continue
        results.append({
            'bbox'    : candidates[idx]['bbox'],
            'class_id': NAME_TO_ID[cls_name],
            'class'   : cls_name,
            'score'   : candidates[idx]['score'],
        })
    return results

print('Gemma 4 fonksiyonu hazır ✅')

In [ ]:
# ── 9. YOLO Label Kaydet ─────────────────────────────────────────────────────
def save_yolo_labels(detections, img_w, img_h, path):
    lines = []
    for d in detections:
        x1, y1, x2, y2 = d['bbox']
        cx = max(0., min(1., ((x1+x2)/2) / img_w))
        cy = max(0., min(1., ((y1+y2)/2) / img_h))
        bw = max(.001, min(1., (x2-x1) / img_w))
        bh = max(.001, min(1., (y2-y1) / img_h))
        lines.append(f"{d['class_id']} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    if lines: Path(path).write_text('\n'.join(lines))
    return len(lines)

In [ ]:
# ── 10. Ana Döngü ────────────────────────────────────────────────────────────
import shutil
from collections import defaultdict

stats = {'labeled':0,'empty':0,'errors':0,'total_det':0,'class_counts':defaultdict(int)}

print(f'🚀 {len(all_images):,} görüntü | SAM3 → Gemma 4 → YOLO')

for i, img_path in enumerate(all_images):
    try:
        image = Image.open(img_path).convert('RGB')
        W, H  = image.size

        candidates = run_sam3(image)
        detections = run_gemma4(image, candidates) if candidates else []

        shutil.copy2(img_path, img_out / img_path.name)
        n = save_yolo_labels(detections, W, H, label_out / (img_path.stem+'.txt'))

        if n > 0:
            stats['labeled']   += 1
            stats['total_det'] += n
            for d in detections: stats['class_counts'][d['class']] += 1
        else:
            stats['empty'] += 1

    except Exception as e:
        stats['errors'] += 1
        print(f'  ❌ [{img_path.name}]: {e}')

    if (i+1) % 10 == 0 or i == len(all_images)-1:
        print(f'  [{i+1}/{len(all_images)}]  labeled={stats["labeled"]}  '
              f'empty={stats["empty"]}  errors={stats["errors"]}')

print('\n✅ Tamamlandı!')

In [ ]:
# ── 11. İstatistikler ────────────────────────────────────────────────────────
total = len(all_images)
print('='*50)
print('📊 SONUÇLAR')
print('='*50)
print(f'Toplam görüntü  : {total:,}')
print(f'Etiketlenen     : {stats["labeled"]:,}  ({stats["labeled"]/max(1,total)*100:.1f}%)')
print(f'Boş             : {stats["empty"]:,}')
print(f'Hata            : {stats["errors"]:,}')
print(f'Toplam tespit   : {stats["total_det"]:,}')
print()
if stats['class_counts']:
    mx = max(stats['class_counts'].values())
    for cls, cnt in sorted(stats['class_counts'].items(), key=lambda x:-x[1]):
        print(f'  {cls:<25} {cnt:>5,}  {"█"*int(cnt/mx*35)}')
print('='*50)

In [ ]:
# ── 12. Görselleştirme ───────────────────────────────────────────────────────
import random, matplotlib.pyplot as plt, matplotlib.patches as patches

PALETTE = ['#00e676','#40c4ff','#ff6d00','#ea80fc','#ffd740','#69f0ae','#ff4081','#80d8ff']

def draw_sample(ax, img_path, lbl_path):
    img = Image.open(img_path); ax.imshow(img); ax.axis('off')
    ax.set_title(img_path.name, fontsize=7, pad=2)
    if not lbl_path.exists() or lbl_path.stat().st_size == 0: return
    W, H = img.size
    for line in lbl_path.read_text().strip().splitlines():
        p = line.split(); cid = int(p[0]); cx,cy,bw,bh = map(float,p[1:5])
        x1,y1 = (cx-bw/2)*W, (cy-bh/2)*H
        c = PALETTE[cid % len(PALETTE)]
        ax.add_patch(patches.Rectangle((x1,y1),bw*W,bh*H,lw=2,edgecolor=c,facecolor='none'))
        ax.text(x1+2,y1-4,CLASS_MAP.get(cid,f'cls{cid}'),color=c,fontsize=7,
                bbox=dict(boxstyle='square,pad=0.1',fc='black',alpha=0.75,lw=0))

lbld = [p for p in img_out.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS
        and (label_out/(p.stem+'.txt')).exists()
        and (label_out/(p.stem+'.txt')).stat().st_size > 0]
sample = random.sample(lbld, min(12,len(lbld)))

if sample:
    cols = min(4,len(sample)); rows = (len(sample)+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*3.5))
    for i, ax in enumerate(np.array(axes).flatten()):
        if i < len(sample): draw_sample(ax, sample[i], label_out/(sample[i].stem+'.txt'))
        else: ax.axis('off')
    plt.suptitle('SAM3 + Gemma 4 — Auto-Label Örnekleri', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'preview.png', dpi=100, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 13. data.yaml + ZIP ──────────────────────────────────────────────────────
import yaml, zipfile, time

yaml_content = {'path':str(OUTPUT_DIR),'train':'images','val':'images',
                'nc':len(TARGET_CLASSES),'names':TARGET_CLASSES}
with open(OUTPUT_DIR/'data.yaml','w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, allow_unicode=True)

zip_path = Path('/kaggle/working') / f'sam3_labeled_{int(time.time())}.zip'
files    = [f for f in OUTPUT_DIR.rglob('*') if f.is_file()]
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as zf:
    for f in files: zf.write(f, f.relative_to(OUTPUT_DIR))

print(f'✅ {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)')
print('→ Kaggle Output sekmesinden indir → unilabellm\'e yükle')

## Parametreler

| Parametre | Varsayılan | Az tespit | Çok tespit |
|-----------|-----------|-----------|------------|
| `SAM_CONF_THRESHOLD` | 0.50 | ↑ 0.65 | ↓ 0.35 |
| `MIN_AREA_RATIO` | 0.001 | ↑ 0.005 | ↓ 0.0005 |
| `MAX_CANDIDATES` | 30 | ↓ 15 | ↑ 50 |